In [1]:
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

# ==========================================
# SETUP & CONFIGURATION
# ==========================================
BANK_FILE = "final_bank_transactions_with_fraud.csv"
OUTPUT_FILE = "cdr_dataset.csv"
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

TOTAL_CALLS = 100000
TOTAL_SUBSCRIBERS = 15000

# Constants for realistic data generation
CITIES = ["Mumbai", "Delhi", "Bengaluru", "Hyderabad", "Chennai", "Kolkata", "Pune", "Ahmedabad", "Jaipur", "Lucknow"]
PROVIDERS = ["Airtel", "Jio", "Vodafone Idea", "BSNL"]
DEVICES = {
    "Samsung Galaxy S24": "Android",
    "iPhone 15": "iOS",
    "OnePlus 12": "Android",
    "Redmi Note 13": "Android",
    "Vivo V30": "Android",
    "iPhone 13": "iOS"
}

def generate_variation(name):
    """Creates slight variations in names for entity resolution testing."""
    if pd.isna(name) or not isinstance(name, str):
        return "Unknown"
    
    parts = name.strip().split()
    if len(parts) == 1:
        return name
    
    variation_type = random.choice(['identical', 'initial', 'lowercase', 'first_only'])
    
    if variation_type == 'identical':
        return name
    elif variation_type == 'initial':
        # Rahul Sharma -> Rahul S.
        return f"{parts[0]} {parts[1][0]}."
    elif variation_type == 'lowercase':
        return name.lower()
    else:
        return parts[0]

def load_or_mock_bank_identities(num_needed):
    """Loads identities from bank data to create overlap, or generates mocks if file is missing."""
    identities = []
    
    if os.path.exists(BANK_FILE):
        print(f"Found {BANK_FILE}. Extracting identities for schema matching...")
        bank_df = pd.read_csv(BANK_FILE)
        
        # Extract Sender Identities
        if 'Sender_Phone_Number' in bank_df.columns:
            subset = bank_df[['Sender_Phone_Number', 'Sender_Customer_Name', 'Sender_IMEI', 'Sender_SIM_Number', 'Sender_City']].dropna().drop_duplicates()
            for _, row in subset.head(num_needed).iterrows():
                identities.append({
                    "MSISDN": str(row['Sender_Phone_Number']).replace('.0', ''),
                    "Name": generate_variation(row['Sender_Customer_Name']),
                    "IMEI": str(row['Sender_IMEI']).replace('.0', ''),
                    "SIM": str(row['Sender_SIM_Number']).replace('.0', ''),
                    "City": row['Sender_City']
                })
    
    # Fill remaining required identities with synthetic data
    remaining = num_needed - len(identities)
    if remaining > 0:
        print(f"Generating {remaining} synthetic identities to reach target subscriber count...")
        for i in range(remaining):
            identities.append({
                "MSISDN": str(random.randint(9000000000, 9999999999)),
                "Name": f"User_{random.randint(1000, 99999)}",
                "IMEI": str(random.randint(100000000000000, 999999999999999)),
                "SIM": f"8991{random.randint(100000000000000, 999999999999999)}",
                "City": random.choice(CITIES)
            })
            
    return pd.DataFrame(identities).sample(frac=1).reset_index(drop=True)

def generate_subscriber_profiles(identities_df):
    """Creates telecom-specific profiles for each identity."""
    print("Building Telecom Subscriber Profiles & Fraud Roles...")
    
    profiles = identities_df.copy()
    profiles['Subscriber_ID'] = [f"SUB_{str(i).zfill(6)}" for i in range(1, len(profiles) + 1)]
    profiles['IMSI'] = [f"404{random.randint(100000000000, 999999999999)}" for _ in range(len(profiles))]
    
    device_models = random.choices(list(DEVICES.keys()), k=len(profiles))
    profiles['Device_Model'] = device_models
    profiles['Device_OS'] = [DEVICES[m] for m in device_models]
    profiles['Network_Provider'] = random.choices(PROVIDERS, k=len(profiles))
    profiles['Network_Type'] = np.random.choice(["4G", "5G"], p=[0.6, 0.4], size=len(profiles))
    
    # Behavioral Baselines
    profiles['Calls_Per_Day'] = np.random.randint(1, 15, size=len(profiles))
    profiles['Average_Call_Duration'] = np.random.randint(30, 300, size=len(profiles))
    profiles['Unique_Contacts_Count'] = np.random.randint(5, 50, size=len(profiles))
    profiles['Night_Call_Ratio'] = np.random.uniform(0.01, 0.15, size=len(profiles)).round(2)
    
    # Assign Fraud Roles (95% Normal, 5% Fraud)
    fraud_types = ["Normal", "SIM_Swap", "Mule_Network", "Burner_Device", "Account_Takeover", "Communication_Fraud"]
    probs = [0.95, 0.01, 0.01, 0.01, 0.01, 0.01]
    
    profiles['Fraud_Type'] = np.random.choice(fraud_types, p=probs, size=len(profiles))
    profiles['Fraud_Flag'] = profiles['Fraud_Type'].apply(lambda x: 0 if x == "Normal" else 1)
    profiles['Investigation_Status'] = profiles['Fraud_Type'].apply(
        lambda x: "Not_Investigated" if x == "Normal" else random.choice(["Under_Review", "Confirmed", "Cleared"])
    )
    
    return profiles

def generate_calls(profiles):
    """Generates 100,000 CDR rows based on subscriber profiles and fraud logic."""
    print(f"Generating {TOTAL_CALLS} Call Detail Records...")
    
    calls = []
    start_date = datetime(2023, 10, 1)
    
    # Pre-calculate arrays for faster random selection
    sub_ids = profiles['Subscriber_ID'].values
    msisdns = profiles['MSISDN'].values
    names = profiles['Name'].values
    
    # Create mappings for quick lookup
    profile_dict = profiles.set_index('Subscriber_ID').to_dict('index')
    
    for i in range(TOTAL_CALLS):
        # Pick Caller
        caller_id = random.choice(sub_ids)
        caller = profile_dict[caller_id]
        
        # Pick Receiver (mostly different from caller)
        receiver_idx = random.randint(0, len(sub_ids)-1)
        while msisdns[receiver_idx] == caller['MSISDN']:
            receiver_idx = random.randint(0, len(sub_ids)-1)
            
        receiver_msisdn = msisdns[receiver_idx]
        receiver_name = names[receiver_idx]
        
        # Base Time & Duration
        call_time = start_date + timedelta(days=random.uniform(0, 30), hours=random.uniform(0, 24))
        duration = int(max(5, random.gauss(caller['Average_Call_Duration'], 50)))
        
        # Assume normal details first
        call_sim = caller['SIM']
        call_imei = caller['IMEI']
        call_city = caller['City'] if random.random() > 0.05 else random.choice(CITIES) # 5% roaming
        
        # FRAUD LOGIC INJECTIONS
        f_type = caller['Fraud_Type']
        
        if f_type == "SIM_Swap":
            # If call is in the second half of the month, simulate the swap
            if call_time.day > 15:
                call_sim = f"8991999{random.randint(100000000000, 999999999999)}"
                call_imei = str(random.randint(100000000000000, 999999999999999))
                
        elif f_type == "Burner_Device":
            # Compress all calls for burner devices into a 2-day window
            call_time = start_date + timedelta(days=random.uniform(1, 3), hours=random.uniform(0, 24))
            duration = int(random.uniform(5, 30)) # Short calls
            
        elif f_type == "Mule_Network":
            # Highly random receivers, short durations
            receiver_msisdn = str(random.randint(9000000000, 9999999999))
            receiver_name = "Unknown"
            
        elif f_type == "Account_Takeover":
            # Device change near specific time
            if call_time.day > 20:
                call_imei = str(random.randint(100000000000000, 999999999999999))
                caller['Night_Call_Ratio'] = 0.8 # Sudden spike in night calls

        # Construct Record
        call_end_time = call_time + timedelta(seconds=duration)
        call_status = np.random.choice(["Completed", "Failed", "Dropped"], p=[0.85, 0.10, 0.05])
        
        if call_status != "Completed":
            duration = int(random.uniform(0, 10))
            
        calls.append({
            "CDR_ID": f"CDR_{str(i+1).zfill(6)}",
            "Call_Start_Time": call_time.strftime("%Y-%m-%d %H:%M:%S"),
            "Call_End_Time": call_end_time.strftime("%Y-%m-%d %H:%M:%S"),
            "Call_Duration_Seconds": duration,
            "Call_Type": random.choice(["Incoming", "Outgoing", "Missed"]),
            "Call_Status": call_status,
            
            "Subscriber_ID": caller_id,
            "Caller_MSISDN": caller['MSISDN'],
            "Receiver_MSISDN": receiver_msisdn,
            "Caller_Name": caller['Name'],
            "Receiver_Name": receiver_name,
            
            "SIM_Number": call_sim,
            "IMSI": caller['IMSI'],
            "IMEI": call_imei,
            "Device_Model": caller['Device_Model'],
            "Device_OS": caller['Device_OS'],
            
            "Network_Provider": caller['Network_Provider'],
            "Network_Type": caller['Network_Type'],
            "Cell_Tower_ID": f"TOWER_{call_city[:3].upper()}_{random.randint(100, 999)}",
            "Tower_City": call_city,
            "Latitude": round(random.uniform(8.0, 37.0), 6), # India Lat bounds approx
            "Longitude": round(random.uniform(68.0, 97.0), 6), # India Lon bounds approx
            
            "Calls_Per_Day": caller['Calls_Per_Day'],
            "Average_Call_Duration": caller['Average_Call_Duration'],
            "Unique_Contacts_Count": caller['Unique_Contacts_Count'],
            "Night_Call_Ratio": caller['Night_Call_Ratio'],
            "International_Call_Flag": 1 if random.random() < 0.01 else 0,
            "Roaming_Flag": 1 if call_city != caller['City'] else 0,
            
            "Fraud_Flag": caller['Fraud_Flag'],
            "Fraud_Type": caller['Fraud_Type'],
            "Investigation_Status": caller['Investigation_Status']
        })

    return pd.DataFrame(calls)

def main():
    print(f"--- Starting CDR Dataset Generation (Target: {TOTAL_CALLS} rows) ---")
    
    # 1. Get Base Identities (Bridging with Bank Data)
    identities_df = load_or_mock_bank_identities(TOTAL_SUBSCRIBERS)
    
    # 2. Build Profiles
    profiles_df = generate_subscriber_profiles(identities_df)
    
    # 3. Generate CDRs
    cdr_df = generate_calls(profiles_df)
    
    # 4. Sort by Timestamp to make it realistic
    cdr_df = cdr_df.sort_values(by="Call_Start_Time").reset_index(drop=True)
    
    # 5. Save Output
    cdr_df.to_csv(OUTPUT_FILE, index=False)
    
    print("\n==========================================")
    print("          GENERATION COMPLETE             ")
    print("==========================================")
    print(f"File Saved: {OUTPUT_FILE}")
    print(f"Total Records: {len(cdr_df)}")
    print(f"Unique Subscribers: {TOTAL_SUBSCRIBERS}")
    print("\n[Fraud Distribution]")
    print(cdr_df['Fraud_Type'].value_counts())
    print("\n[Columns Created]")
    print(cdr_df.columns.tolist())
    print("\nReady for Schema Matching and Graph Fusion!")

if __name__ == "__main__":
    main()

--- Starting CDR Dataset Generation (Target: 100000 rows) ---
Found final_bank_transactions_with_fraud.csv. Extracting identities for schema matching...
Building Telecom Subscriber Profiles & Fraud Roles...
Generating 100000 Call Detail Records...

          GENERATION COMPLETE             
File Saved: cdr_dataset.csv
Total Records: 100000
Unique Subscribers: 15000

[Fraud Distribution]
Fraud_Type
Normal                 94969
Communication_Fraud     1084
Burner_Device           1057
Account_Takeover         991
Mule_Network             988
SIM_Swap                 911
Name: count, dtype: int64

[Columns Created]
['CDR_ID', 'Call_Start_Time', 'Call_End_Time', 'Call_Duration_Seconds', 'Call_Type', 'Call_Status', 'Subscriber_ID', 'Caller_MSISDN', 'Receiver_MSISDN', 'Caller_Name', 'Receiver_Name', 'SIM_Number', 'IMSI', 'IMEI', 'Device_Model', 'Device_OS', 'Network_Provider', 'Network_Type', 'Cell_Tower_ID', 'Tower_City', 'Latitude', 'Longitude', 'Calls_Per_Day', 'Average_Call_Duration', 'U

In [2]:
## validation script:

In [3]:
import pandas as pd
import os

FILE_PATH = "cdr_dataset.csv"

# ==========================================
# LOAD DATASET
# ==========================================

if not os.path.exists(FILE_PATH):
    print(f"File {FILE_PATH} not found")
    exit()

df = pd.read_csv(FILE_PATH)

print("="*80)
print("CDR DATASET BASIC INFORMATION")
print("="*80)

print("\nShape:")
print(df.shape)


# ==========================================
# COLUMN CHECK
# ==========================================

print("\n" + "="*80)
print("COLUMN NAMES")
print("="*80)

for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")


# ==========================================
# DATA TYPES
# ==========================================

print("\n" + "="*80)
print("DATA TYPES")
print("="*80)

print(df.dtypes)


# ==========================================
# FIRST ROWS
# ==========================================

print("\n" + "="*80)
print("FIRST 5 ROWS")
print("="*80)

print(df.head())


# ==========================================
# LAST ROWS
# ==========================================

print("\n" + "="*80)
print("LAST 5 ROWS")
print("="*80)

print(df.tail())


# ==========================================
# MISSING VALUES
# ==========================================

print("\n" + "="*80)
print("MISSING VALUE CHECK")
print("="*80)

missing = df.isnull().sum()

print(missing[missing > 0])

if missing.sum() == 0:
    print("No missing values")


# ==========================================
# UNIQUE COUNTS
# ==========================================

print("\n" + "="*80)
print("UNIQUE VALUE COUNTS")
print("="*80)

for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")


# ==========================================
# DUPLICATE CHECK
# ==========================================

print("\n" + "="*80)
print("DUPLICATE CHECK")
print("="*80)

print("Duplicate rows:", df.duplicated().sum())


# ==========================================
# IMPORTANT COLUMN VALUE CHECKS
# ==========================================

important_columns = [
    "CDR_ID",
    "Subscriber_ID",
    "Caller_MSISDN",
    "Receiver_MSISDN",
    "IMEI",
    "SIM_Number",
    "Call_Type",
    "Call_Status",
    "Network_Type",
    "Fraud_Flag",
    "Fraud_Type"
]


print("\n" + "="*80)
print("IMPORTANT COLUMN SAMPLES")
print("="*80)

for col in important_columns:
    if col in df.columns:
        print("\n", col)
        print(df[col].head(10).tolist())


# ==========================================
# FRAUD DISTRIBUTION
# ==========================================

print("\n" + "="*80)
print("FRAUD DISTRIBUTION")
print("="*80)

for col in ["Fraud_Flag", "Fraud_Type"]:
    if col in df.columns:
        print("\n", col)
        print(df[col].value_counts())


# ==========================================
# CALL STATISTICS
# ==========================================

print("\n" + "="*80)
print("CALL STATISTICS")
print("="*80)

numeric_cols = [
    "Call_Duration_Seconds",
    "Calls_Per_Day",
    "Average_Call_Duration",
    "Unique_Contacts_Count"
]

for col in numeric_cols:
    if col in df.columns:
        print("\n", col)
        print(df[col].describe())


# ==========================================
# PHONE FORMAT CHECK
# ==========================================

print("\n" + "="*80)
print("PHONE FORMAT CHECK")
print("="*80)

for col in ["Caller_MSISDN", "Receiver_MSISDN"]:
    if col in df.columns:
        lengths = df[col].astype(str).str.len().value_counts()
        print("\n", col)
        print(lengths.head())


# ==========================================
# IMEI CHECK
# ==========================================

print("\n" + "="*80)
print("IMEI LENGTH CHECK")
print("="*80)

if "IMEI" in df.columns:
    print(df["IMEI"].astype(str).str.len().value_counts())


print("\n" + "="*80)
print("CHECK COMPLETE")
print("="*80)

CDR DATASET BASIC INFORMATION

Shape:
(100000, 31)

COLUMN NAMES
1. CDR_ID
2. Call_Start_Time
3. Call_End_Time
4. Call_Duration_Seconds
5. Call_Type
6. Call_Status
7. Subscriber_ID
8. Caller_MSISDN
9. Receiver_MSISDN
10. Caller_Name
11. Receiver_Name
12. SIM_Number
13. IMSI
14. IMEI
15. Device_Model
16. Device_OS
17. Network_Provider
18. Network_Type
19. Cell_Tower_ID
20. Tower_City
21. Latitude
22. Longitude
23. Calls_Per_Day
24. Average_Call_Duration
25. Unique_Contacts_Count
26. Night_Call_Ratio
27. International_Call_Flag
28. Roaming_Flag
29. Fraud_Flag
30. Fraud_Type
31. Investigation_Status

DATA TYPES
CDR_ID                      object
Call_Start_Time             object
Call_End_Time               object
Call_Duration_Seconds        int64
Call_Type                   object
Call_Status                 object
Subscriber_ID               object
Caller_MSISDN                int64
Receiver_MSISDN              int64
Caller_Name                 object
Receiver_Name               object

In [4]:
##correction script:
import pandas as pd

df = pd.read_csv("cdr_dataset.csv")

string_columns = [
    "Caller_MSISDN",
    "Receiver_MSISDN",
    "SIM_Number",
    "IMSI",
    "IMEI"
]

for col in string_columns:
    if col in df.columns:
        df[col] = df[col].astype(str)

df.to_csv("cdr_dataset.csv", index=False)

print("Identifier columns converted to string")

Identifier columns converted to string


In [5]:
for col1, col2 in [
    ("Caller_MSISDN","Subscriber_ID"),
    ("Caller_MSISDN","Caller_Name"),
    ("Receiver_MSISDN","Receiver_Name")
]:

    print("\nChecking", col1, "->", col2)

    check = df.groupby(col1)[col2].nunique()

    bad = check[check > 1]

    if len(bad)==0:
        print("PASS")
    else:
        print("FAILED:", len(bad))
        print(bad.head())


Checking Caller_MSISDN -> Subscriber_ID
FAILED: 1
Caller_MSISDN
9465101586    2
Name: Subscriber_ID, dtype: int64

Checking Caller_MSISDN -> Caller_Name
FAILED: 1
Caller_MSISDN
9465101586    2
Name: Caller_Name, dtype: int64

Checking Receiver_MSISDN -> Receiver_Name
FAILED: 1
Receiver_MSISDN
9465101586    2
Name: Receiver_Name, dtype: int64


In [7]:
import pandas as pd

FILE = "your_cdr_file.csv"

df = pd.read_csv('cdr_dataset.csv')

bad_number = 9465101586

print(df[df["Caller_MSISDN"] == bad_number][
    ["Caller_MSISDN","Subscriber_ID","Caller_Name"]
])

# choose first occurrence as canonical
canonical = df[df["Caller_MSISDN"] == bad_number].iloc[0]

canonical_subscriber = canonical["Subscriber_ID"]
canonical_name = canonical["Caller_Name"]

# Fix caller side
mask = df["Caller_MSISDN"] == bad_number

df.loc[mask, "Subscriber_ID"] = canonical_subscriber
df.loc[mask, "Caller_Name"] = canonical_name


# Fix receiver side
mask_receiver = df["Receiver_MSISDN"] == bad_number

df.loc[mask_receiver, "Receiver_Name"] = canonical_name


df.to_csv(FILE, index=False)

print("Fixed successfully")

       Caller_MSISDN Subscriber_ID     Caller_Name
386       9465101586    SUB_007185  Gayatri Behera
8874      9465101586    SUB_007185  Gayatri Behera
22174     9465101586    SUB_007185  Gayatri Behera
22278     9465101586    SUB_007185  Gayatri Behera
25499     9465101586    SUB_007185  Gayatri Behera
30300     9465101586    SUB_010700        Nisha N.
31389     9465101586    SUB_007185  Gayatri Behera
52432     9465101586    SUB_007185  Gayatri Behera
79436     9465101586    SUB_007185  Gayatri Behera
83410     9465101586    SUB_010700        Nisha N.
Fixed successfully


In [8]:
for col1, col2 in [
    ("Caller_MSISDN","Subscriber_ID"),
    ("Caller_MSISDN","Caller_Name"),
    ("Receiver_MSISDN","Receiver_Name")
]:

    print("\nChecking", col1, "->", col2)

    bad = df.groupby(col1)[col2].nunique()
    bad = bad[bad > 1]

    print("FAILED:", len(bad))


Checking Caller_MSISDN -> Subscriber_ID
FAILED: 0

Checking Caller_MSISDN -> Caller_Name
FAILED: 0

Checking Receiver_MSISDN -> Receiver_Name
FAILED: 0
